# Chapter 8.7: Testing & Validating Model Correctness

## Learning Objectives

By the end of this notebook, you will:

1. **Design** a testing strategy for new model implementations
2. **Implement** numerical correctness tests against HuggingFace
3. **Set** appropriate tolerance thresholds for different precisions
4. **Write** greedy decoding comparison tests
5. **Evaluate** generation quality
6. **Handle** edge cases: empty input, max length, special tokens
7. **Compare** perplexity between implementations
8. **Build** an automated test framework

---

## Prerequisites
- Chapters 8.1-8.6 (Model implementation)
- Understanding of floating-point precision

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part8/chapter_8.7_testing.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part8/chapter_8.7_testing.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Testing Strategy Overview

```
Testing Pyramid for Model Implementations
==========================================

            /\          Perplexity & Quality Tests
           /  \         (slowest, most comprehensive)
          /    \
         /------\       Generation Tests
        /        \      (greedy decode comparison)
       /----------\
      /            \    Output Comparison Tests
     /              \   (logit-level comparison with HF)
    /----------------\
   /                  \ Weight Loading Tests
  /                    \ (correct shapes, values)
 /--------------------\
/                      \ Unit Tests
/________________________\ (individual layers)
```

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from typing import List, Tuple, Optional, Dict
import time

# Test categories
test_categories = {
    "Unit Tests": {
        "what": "Test individual layers (MLP, attention, norm)",
        "speed": "Fast (< 1 second)",
        "when": "During development, on every change",
        "examples": [
            "MLP output shape matches expected",
            "Attention handles GQA correctly",
            "RoPE produces correct rotations",
            "Layer norm output has zero mean",
        ],
    },
    "Weight Loading Tests": {
        "what": "Verify weights are loaded correctly from HF checkpoint",
        "speed": "Medium (seconds to minutes)",
        "when": "After implementing load_weights()",
        "examples": [
            "All HF weights have corresponding parameters",
            "Weight shapes match",
            "Merged weights (QKV) are correct",
            "No unexpected zero tensors",
        ],
    },
    "Output Comparison Tests": {
        "what": "Compare logits between vLLM/SGLang and HuggingFace",
        "speed": "Medium (seconds)",
        "when": "After complete model implementation",
        "examples": [
            "Logits match within tolerance",
            "Top-k token predictions match",
            "Cosine similarity > 0.99",
        ],
    },
    "Generation Tests": {
        "what": "Compare generated text between implementations",
        "speed": "Slow (seconds to minutes)",
        "when": "Before merging/deploying",
        "examples": [
            "Greedy decode produces identical tokens",
            "Beam search results match",
            "Stop conditions work correctly",
        ],
    },
    "Quality Tests": {
        "what": "Perplexity, benchmark scores, generation quality",
        "speed": "Very slow (minutes to hours)",
        "when": "Final validation before release",
        "examples": [
            "Perplexity matches HF within 0.1%",
            "MMLU score matches",
            "HumanEval score matches",
        ],
    },
}

for category, info in test_categories.items():
    print(f"\n{category}")
    print(f"  What: {info['what']}")
    print(f"  Speed: {info['speed']}")
    print(f"  When: {info['when']}")
    print(f"  Examples:")
    for ex in info['examples']:
        print(f"    - {ex}")

## 2. Tolerance Thresholds for Different Precisions

Different numerical precisions produce different levels of error. Setting the right tolerance is crucial.

In [ ]:
# Precision and tolerance reference

precisions = {
    "FP32": {
        "bits": 32,
        "mantissa_bits": 23,
        "machine_epsilon": 1.19e-7,
        "recommended_atol": 1e-5,
        "recommended_rtol": 1e-5,
        "typical_logit_diff": "< 1e-4",
    },
    "FP16": {
        "bits": 16,
        "mantissa_bits": 10,
        "machine_epsilon": 9.77e-4,
        "recommended_atol": 1e-3,
        "recommended_rtol": 1e-3,
        "typical_logit_diff": "< 1e-2",
    },
    "BF16": {
        "bits": 16,
        "mantissa_bits": 7,
        "machine_epsilon": 3.91e-3,
        "recommended_atol": 1e-2,
        "recommended_rtol": 1e-2,
        "typical_logit_diff": "< 5e-2",
    },
    "FP8 (E4M3)": {
        "bits": 8,
        "mantissa_bits": 3,
        "machine_epsilon": 6.25e-2,
        "recommended_atol": 0.1,
        "recommended_rtol": 0.1,
        "typical_logit_diff": "< 0.5",
    },
    "INT4 (GPTQ/AWQ)": {
        "bits": 4,
        "mantissa_bits": "N/A",
        "machine_epsilon": "N/A",
        "recommended_atol": 0.5,
        "recommended_rtol": 0.2,
        "typical_logit_diff": "< 1.0",
    },
}

print(f"{'Precision':<15} {'atol':<10} {'rtol':<10} {'Typical Logit Diff':<20}")
print("-" * 55)
for name, info in precisions.items():
    print(f"{name:<15} {info['recommended_atol']:<10} {info['recommended_rtol']:<10} {info['typical_logit_diff']:<20}")

print("\nNote: atol = absolute tolerance, rtol = relative tolerance")
print("Use: torch.allclose(a, b, atol=atol, rtol=rtol)")

## 3. Unit Tests for Individual Layers

In [ ]:
class LayerTestSuite:
    """Unit tests for individual model layers."""
    
    def __init__(self, hidden_size=256, num_heads=4, num_kv_heads=2, intermediate_size=512):
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.num_kv_heads = num_kv_heads
        self.head_dim = hidden_size // num_heads
        self.intermediate_size = intermediate_size
        self.results = []
    
    def _report(self, test_name, passed, details=""):
        status = "PASS" if passed else "FAIL"
        self.results.append((test_name, passed))
        print(f"  [{status}] {test_name}", end="")
        if details:
            print(f" - {details}")
        else:
            print()
    
    def test_mlp_shapes(self):
        """Test MLP input/output shapes."""
        print("\nTest: MLP Shapes")
        mlp = nn.Sequential(
            nn.Linear(self.hidden_size, 2 * self.intermediate_size, bias=False),
        )
        
        for batch_size in [1, 4, 16]:
            x = torch.randn(batch_size, self.hidden_size)
            out = mlp(x)
            expected_shape = (batch_size, 2 * self.intermediate_size)
            passed = out.shape == expected_shape
            self._report(
                f"MLP shape batch={batch_size}",
                passed,
                f"got {out.shape}, expected {expected_shape}"
            )
    
    def test_qkv_split(self):
        """Test Q/K/V splitting for GQA."""
        print("\nTest: QKV Split")
        
        q_size = self.num_heads * self.head_dim
        kv_size = self.num_kv_heads * self.head_dim
        total_size = q_size + 2 * kv_size
        
        qkv = torch.randn(8, total_size)
        q = qkv[:, :q_size]
        k = qkv[:, q_size:q_size + kv_size]
        v = qkv[:, q_size + kv_size:]
        
        self._report(
            "Q shape",
            q.shape == (8, q_size),
            f"Q: {q.shape}, expected (8, {q_size})"
        )
        self._report(
            "K shape",
            k.shape == (8, kv_size),
            f"K: {k.shape}, expected (8, {kv_size})"
        )
        self._report(
            "V shape",
            v.shape == (8, kv_size),
            f"V: {v.shape}, expected (8, {kv_size})"
        )
        self._report(
            "GQA ratio",
            self.num_heads % self.num_kv_heads == 0,
            f"{self.num_heads}/{self.num_kv_heads} = {self.num_heads//self.num_kv_heads}"
        )
    
    def test_rms_norm(self):
        """Test RMS normalization properties."""
        print("\nTest: RMS Normalization")
        
        eps = 1e-6
        x = torch.randn(4, 8, self.hidden_size)
        
        # Manual RMS norm
        rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True) + eps)
        normalized = x / rms
        
        # Check RMS is approximately 1
        output_rms = torch.sqrt(normalized.pow(2).mean(dim=-1))
        self._report(
            "RMS ~= 1 after normalization",
            torch.allclose(output_rms, torch.ones_like(output_rms), atol=0.1),
            f"mean RMS: {output_rms.mean():.4f}"
        )
    
    def test_rope_properties(self):
        """Test Rotary Position Embedding properties."""
        print("\nTest: RoPE Properties")
        
        # RoPE should preserve norm
        head_dim = self.head_dim
        x = torch.randn(8, head_dim)
        
        # Simple rotation matrix
        theta = 10000.0
        freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
        positions = torch.arange(8).float()
        angles = positions.unsqueeze(1) * freqs.unsqueeze(0)
        cos = torch.cos(angles)
        sin = torch.sin(angles)
        
        # Apply rotation
        x1 = x[:, 0::2]
        x2 = x[:, 1::2]
        rotated_x1 = x1 * cos - x2 * sin
        rotated_x2 = x1 * sin + x2 * cos
        rotated = torch.stack([rotated_x1, rotated_x2], dim=-1).flatten(-2)
        
        # Norms should be preserved
        orig_norm = x.norm(dim=-1)
        rot_norm = rotated.norm(dim=-1)
        
        self._report(
            "RoPE preserves norm",
            torch.allclose(orig_norm, rot_norm, atol=1e-5),
            f"max norm diff: {(orig_norm - rot_norm).abs().max():.6f}"
        )
    
    def summary(self):
        """Print test summary."""
        passed = sum(1 for _, p in self.results if p)
        total = len(self.results)
        print(f"\n{'='*40}")
        print(f"Results: {passed}/{total} tests passed")
        if passed == total:
            print("All tests passed!")
        else:
            failed = [name for name, p in self.results if not p]
            print(f"Failed tests: {failed}")

# Run unit tests
suite = LayerTestSuite()
suite.test_mlp_shapes()
suite.test_qkv_split()
suite.test_rms_norm()
suite.test_rope_properties()
suite.summary()

## 4. Weight Loading Tests

In [ ]:
def test_weight_loading(model, hf_state_dict: Dict[str, torch.Tensor]):
    """
    Test that weights are correctly loaded from HuggingFace format.
    
    Checks:
    1. All HF weights are accounted for
    2. All model parameters are initialized
    3. No parameters are all zeros (likely unloaded)
    4. Shapes are compatible
    """
    print("Weight Loading Tests")
    print("=" * 50)
    
    model_params = dict(model.named_parameters())
    hf_names = set(hf_state_dict.keys())
    model_names = set(model_params.keys())
    
    # Test 1: Check for unmatched HF weights
    # Some weights are expected to be skipped (e.g., rotary_emb.inv_freq)
    skip_patterns = ["rotary_emb.inv_freq", "position_ids"]
    
    unmatched_hf = []
    for name in hf_names:
        if any(pattern in name for pattern in skip_patterns):
            continue
        # Check if this weight maps to a model parameter
        # (accounting for merged weights like qkv_proj)
        found = False
        for model_name in model_names:
            if name in model_name or any(
                name.replace(old, new) in model_name
                for old, new in [("q_proj", "qkv_proj"), ("k_proj", "qkv_proj"),
                                  ("v_proj", "qkv_proj"), ("gate_proj", "gate_up_proj"),
                                  ("up_proj", "gate_up_proj")]
            ):
                found = True
                break
        if not found:
            unmatched_hf.append(name)
    
    passed = len(unmatched_hf) == 0
    print(f"  [{'PASS' if passed else 'WARN'}] Unmatched HF weights: {len(unmatched_hf)}")
    for name in unmatched_hf[:5]:
        print(f"    - {name}")
    
    # Test 2: Check for zero parameters
    zero_params = []
    for name, param in model_params.items():
        if param.abs().sum() == 0:
            zero_params.append(name)
    
    passed = len(zero_params) == 0
    print(f"  [{'PASS' if passed else 'WARN'}] Zero parameters: {len(zero_params)}")
    for name in zero_params[:5]:
        print(f"    - {name}")
    
    # Test 3: Parameter statistics
    print(f"\n  Parameter statistics:")
    total_params = sum(p.numel() for p in model.parameters())
    print(f"    Total parameters: {total_params:,}")
    
    for name, param in list(model_params.items())[:5]:
        print(f"    {name}: shape={list(param.shape)}, "
              f"mean={param.float().mean():.4f}, std={param.float().std():.4f}")

# Demo
simple_model = nn.Sequential(
    nn.Linear(256, 256),
    nn.Linear(256, 1000),
)

fake_hf_weights = {
    "0.weight": torch.randn(256, 256),
    "0.bias": torch.randn(256),
    "1.weight": torch.randn(1000, 256),
    "1.bias": torch.randn(1000),
}

# Load weights
simple_model.load_state_dict(fake_hf_weights)
test_weight_loading(simple_model, fake_hf_weights)

## 5. Numerical Correctness: Logit Comparison

In [ ]:
def compare_logits(
    logits_a: torch.Tensor,
    logits_b: torch.Tensor,
    label_a: str = "Implementation A",
    label_b: str = "Implementation B",
    atol: float = 1e-3,
    rtol: float = 1e-3,
) -> Dict[str, float]:
    """
    Compare logits from two implementations.
    
    Returns a dict of metrics.
    """
    print(f"\nComparing {label_a} vs {label_b}")
    print("=" * 50)
    
    a = logits_a.float()
    b = logits_b.float()
    
    # Basic shape check
    assert a.shape == b.shape, f"Shape mismatch: {a.shape} vs {b.shape}"
    print(f"Shape: {a.shape}")
    
    # Absolute difference
    abs_diff = (a - b).abs()
    max_abs_diff = abs_diff.max().item()
    mean_abs_diff = abs_diff.mean().item()
    
    print(f"\nAbsolute difference:")
    print(f"  Max:  {max_abs_diff:.6f}")
    print(f"  Mean: {mean_abs_diff:.6f}")
    
    # Relative difference
    rel_diff = abs_diff / (a.abs() + 1e-8)
    max_rel_diff = rel_diff.max().item()
    mean_rel_diff = rel_diff.mean().item()
    
    print(f"\nRelative difference:")
    print(f"  Max:  {max_rel_diff:.6f}")
    print(f"  Mean: {mean_rel_diff:.6f}")
    
    # Cosine similarity
    cos_sim = torch.nn.functional.cosine_similarity(
        a.flatten().unsqueeze(0),
        b.flatten().unsqueeze(0)
    ).item()
    print(f"\nCosine similarity: {cos_sim:.8f}")
    
    # Top-k agreement
    for k in [1, 5, 10]:
        topk_a = a.topk(k, dim=-1).indices
        topk_b = b.topk(k, dim=-1).indices
        agreement = (topk_a == topk_b).float().mean().item()
        print(f"  Top-{k} agreement: {agreement:.4%}")
    
    # torch.allclose test
    is_close = torch.allclose(a, b, atol=atol, rtol=rtol)
    print(f"\ntorch.allclose(atol={atol}, rtol={rtol}): {is_close}")
    
    return {
        "max_abs_diff": max_abs_diff,
        "mean_abs_diff": mean_abs_diff,
        "cosine_similarity": cos_sim,
        "is_close": is_close,
    }

# Demo: Compare two similar tensors
logits_hf = torch.randn(1, 32, 32000)  # [batch, seq, vocab]
logits_vllm = logits_hf + torch.randn_like(logits_hf) * 0.001  # Small perturbation

metrics = compare_logits(logits_hf, logits_vllm, "HuggingFace", "vLLM")

## 6. Greedy Decoding Comparison

In [ ]:
def greedy_decode(
    model: nn.Module,
    input_ids: torch.Tensor,
    max_new_tokens: int = 20,
    vocab_size: int = 1000,
) -> List[int]:
    """
    Simple greedy decoding for testing.
    """
    generated = input_ids.tolist() if input_ids.dim() == 1 else input_ids[0].tolist()
    
    for _ in range(max_new_tokens):
        input_tensor = torch.tensor([generated])
        with torch.no_grad():
            logits = model(input_tensor)
        
        # Get the last token's logits
        next_token_logits = logits[0, -1, :]
        next_token = next_token_logits.argmax().item()
        generated.append(next_token)
        
        # Stop on EOS (token 2)
        if next_token == 2:
            break
    
    return generated


def compare_greedy_decoding(
    model_a: nn.Module,
    model_b: nn.Module,
    input_ids: torch.Tensor,
    max_new_tokens: int = 20,
    label_a: str = "Model A",
    label_b: str = "Model B",
):
    """
    Compare greedy decoding outputs of two models.
    """
    print(f"\nGreedy Decoding Comparison")
    print("=" * 50)
    
    tokens_a = greedy_decode(model_a, input_ids, max_new_tokens)
    tokens_b = greedy_decode(model_b, input_ids, max_new_tokens)
    
    print(f"Input tokens: {input_ids.tolist()}")
    print(f"{label_a}: {tokens_a}")
    print(f"{label_b}: {tokens_b}")
    
    # Compare
    min_len = min(len(tokens_a), len(tokens_b))
    matching = sum(1 for a, b in zip(tokens_a, tokens_b) if a == b)
    
    print(f"\nMatching tokens: {matching}/{min_len} ({matching/min_len:.1%})")
    
    # Find first divergence
    for i, (a, b) in enumerate(zip(tokens_a, tokens_b)):
        if a != b:
            print(f"First divergence at position {i}: {a} vs {b}")
            break
    else:
        if len(tokens_a) == len(tokens_b):
            print("Outputs are IDENTICAL!")
    
    return tokens_a == tokens_b

# Demo with two identical models
model_a = nn.Sequential(
    nn.Embedding(100, 64),
    nn.Linear(64, 100),
)
model_b = nn.Sequential(
    nn.Embedding(100, 64),
    nn.Linear(64, 100),
)
# Copy weights
model_b.load_state_dict(model_a.state_dict())

input_ids = torch.tensor([1, 5, 10])
identical = compare_greedy_decoding(model_a, model_b, input_ids, max_new_tokens=10, 
                                   label_a="Original", label_b="Copy")

## 7. Edge Case Tests

In [ ]:
def test_edge_cases(model: nn.Module, vocab_size: int, hidden_size: int):
    """
    Test edge cases that often reveal bugs.
    """
    print("Edge Case Tests")
    print("=" * 50)
    results = []
    
    # Test 1: Single token input
    print("\nTest 1: Single token input")
    try:
        x = torch.tensor([[1]])
        with torch.no_grad():
            out = model(x)
        passed = out.shape[-1] == vocab_size
        results.append(("Single token", passed))
        print(f"  [{'PASS' if passed else 'FAIL'}] Output shape: {out.shape}")
    except Exception as e:
        results.append(("Single token", False))
        print(f"  [FAIL] Error: {e}")
    
    # Test 2: Long sequence
    print("\nTest 2: Long sequence (512 tokens)")
    try:
        x = torch.randint(0, vocab_size, (1, 512))
        with torch.no_grad():
            out = model(x)
        passed = out.shape == (1, 512, vocab_size)
        results.append(("Long sequence", passed))
        print(f"  [{'PASS' if passed else 'FAIL'}] Output shape: {out.shape}")
    except Exception as e:
        results.append(("Long sequence", False))
        print(f"  [FAIL] Error: {e}")
    
    # Test 3: Special token IDs (0 = pad, 1 = BOS, 2 = EOS)
    print("\nTest 3: Special tokens")
    try:
        x = torch.tensor([[0, 1, 2, 0, 0]])  # pad, BOS, EOS, pad, pad
        with torch.no_grad():
            out = model(x)
        passed = not torch.isnan(out).any() and not torch.isinf(out).any()
        results.append(("Special tokens", passed))
        print(f"  [{'PASS' if passed else 'FAIL'}] No NaN/Inf: {passed}")
    except Exception as e:
        results.append(("Special tokens", False))
        print(f"  [FAIL] Error: {e}")
    
    # Test 4: Max token ID
    print("\nTest 4: Max token ID (vocab_size - 1)")
    try:
        x = torch.tensor([[vocab_size - 1]])
        with torch.no_grad():
            out = model(x)
        passed = not torch.isnan(out).any()
        results.append(("Max token ID", passed))
        print(f"  [{'PASS' if passed else 'FAIL'}] No NaN: {passed}")
    except Exception as e:
        results.append(("Max token ID", False))
        print(f"  [FAIL] Error: {e}")
    
    # Test 5: Determinism (same input -> same output)
    print("\nTest 5: Determinism")
    try:
        x = torch.randint(0, vocab_size, (1, 16))
        with torch.no_grad():
            out1 = model(x)
            out2 = model(x)
        passed = torch.equal(out1, out2)
        results.append(("Determinism", passed))
        print(f"  [{'PASS' if passed else 'FAIL'}] Outputs identical: {passed}")
    except Exception as e:
        results.append(("Determinism", False))
        print(f"  [FAIL] Error: {e}")
    
    # Test 6: No NaN in outputs for random inputs
    print("\nTest 6: NaN stability (100 random inputs)")
    nan_count = 0
    for _ in range(100):
        x = torch.randint(0, vocab_size, (1, np.random.randint(1, 64)))
        with torch.no_grad():
            out = model(x)
        if torch.isnan(out).any():
            nan_count += 1
    passed = nan_count == 0
    results.append(("NaN stability", passed))
    print(f"  [{'PASS' if passed else 'FAIL'}] NaN occurrences: {nan_count}/100")
    
    # Summary
    passed_total = sum(1 for _, p in results if p)
    print(f"\nEdge case results: {passed_total}/{len(results)} passed")
    return results

# Run edge case tests
test_model = nn.Sequential(
    nn.Embedding(1000, 256),
    nn.LayerNorm(256),
    nn.Linear(256, 1000),
)
test_edge_cases(test_model, vocab_size=1000, hidden_size=256)

## 8. Perplexity Comparison

In [ ]:
def compute_perplexity(
    model: nn.Module,
    text_ids: torch.Tensor,  # [1, seq_len]
    stride: int = 512,
) -> float:
    """
    Compute perplexity of a model on given text.
    
    Uses sliding window with stride for long texts.
    """
    model.eval()
    nlls = []
    seq_len = text_ids.shape[1]
    
    for begin in range(0, seq_len - 1, stride):
        end = min(begin + stride + 1, seq_len)
        input_ids = text_ids[:, begin:end]
        target_ids = input_ids[:, 1:].clone()
        
        with torch.no_grad():
            logits = model(input_ids)
        
        # Compute negative log-likelihood
        logits = logits[:, :-1, :]  # Shift logits
        loss_fn = nn.CrossEntropyLoss(reduction='none')
        nll = loss_fn(
            logits.reshape(-1, logits.shape[-1]),
            target_ids.reshape(-1)
        )
        nlls.append(nll)
    
    # Compute perplexity
    all_nlls = torch.cat(nlls)
    ppl = torch.exp(all_nlls.mean()).item()
    
    return ppl


def compare_perplexity(
    model_a: nn.Module,
    model_b: nn.Module,
    text_ids: torch.Tensor,
    label_a: str = "Model A",
    label_b: str = "Model B",
    tolerance: float = 0.01,  # 1% tolerance
):
    """
    Compare perplexity between two model implementations.
    """
    print(f"\nPerplexity Comparison")
    print("=" * 50)
    
    ppl_a = compute_perplexity(model_a, text_ids)
    ppl_b = compute_perplexity(model_b, text_ids)
    
    rel_diff = abs(ppl_a - ppl_b) / ppl_a
    
    print(f"  {label_a} perplexity: {ppl_a:.4f}")
    print(f"  {label_b} perplexity: {ppl_b:.4f}")
    print(f"  Relative difference: {rel_diff:.4%}")
    print(f"  Tolerance: {tolerance:.4%}")
    
    passed = rel_diff < tolerance
    print(f"  Result: {'PASS' if passed else 'FAIL'}")
    
    return passed, ppl_a, ppl_b

# Demo
model_a = nn.Sequential(nn.Embedding(100, 64), nn.Linear(64, 100))
model_b = nn.Sequential(nn.Embedding(100, 64), nn.Linear(64, 100))
model_b.load_state_dict(model_a.state_dict())

text = torch.randint(0, 100, (1, 200))
passed, ppl_a, ppl_b = compare_perplexity(model_a, model_b, text, "Original", "Copy")

## 9. Automated Test Framework

In [ ]:
class ModelTestFramework:
    """
    Comprehensive test framework for model implementations.
    
    Usage:
        framework = ModelTestFramework(model, config)
        framework.run_all()
    """
    
    def __init__(self, model, config, reference_model=None):
        self.model = model
        self.config = config
        self.reference_model = reference_model
        self.results = {}
    
    def test_forward_shapes(self):
        """Test that forward pass produces correct shapes."""
        print("\n1. Forward Shape Tests")
        print("-" * 40)
        passed = True
        
        for seq_len in [1, 8, 32, 128]:
            x = torch.randint(0, self.config.vocab_size, (1, seq_len))
            with torch.no_grad():
                out = self.model(x)
            expected = (1, seq_len, self.config.vocab_size)
            ok = out.shape == expected
            if not ok:
                passed = False
            print(f"  seq_len={seq_len}: {out.shape} {'OK' if ok else 'FAIL'}")
        
        self.results['forward_shapes'] = passed
    
    def test_numerical_stability(self):
        """Test for NaN/Inf in outputs."""
        print("\n2. Numerical Stability Tests")
        print("-" * 40)
        
        nan_count = 0
        inf_count = 0
        n_tests = 50
        
        for _ in range(n_tests):
            seq_len = np.random.randint(1, 128)
            x = torch.randint(0, self.config.vocab_size, (1, seq_len))
            with torch.no_grad():
                out = self.model(x)
            if torch.isnan(out).any():
                nan_count += 1
            if torch.isinf(out).any():
                inf_count += 1
        
        passed = nan_count == 0 and inf_count == 0
        print(f"  NaN occurrences: {nan_count}/{n_tests}")
        print(f"  Inf occurrences: {inf_count}/{n_tests}")
        print(f"  Result: {'PASS' if passed else 'FAIL'}")
        self.results['numerical_stability'] = passed
    
    def test_determinism(self):
        """Test output determinism."""
        print("\n3. Determinism Tests")
        print("-" * 40)
        
        x = torch.randint(0, self.config.vocab_size, (1, 32))
        with torch.no_grad():
            out1 = self.model(x)
            out2 = self.model(x)
        
        passed = torch.equal(out1, out2)
        print(f"  Same input -> same output: {passed}")
        if not passed:
            diff = (out1 - out2).abs().max().item()
            print(f"  Max difference: {diff}")
        self.results['determinism'] = passed
    
    def test_reference_comparison(self):
        """Compare with reference model."""
        if self.reference_model is None:
            print("\n4. Reference Comparison: SKIPPED (no reference model)")
            return
        
        print("\n4. Reference Comparison Tests")
        print("-" * 40)
        
        x = torch.randint(0, self.config.vocab_size, (1, 32))
        with torch.no_grad():
            out_model = self.model(x)
            out_ref = self.reference_model(x)
        
        metrics = compare_logits(out_model, out_ref, "Model", "Reference")
        self.results['reference_comparison'] = metrics['is_close']
    
    def run_all(self):
        """Run all tests."""
        print("\n" + "=" * 60)
        print("MODEL TEST FRAMEWORK")
        print("=" * 60)
        
        self.test_forward_shapes()
        self.test_numerical_stability()
        self.test_determinism()
        self.test_reference_comparison()
        
        # Summary
        print("\n" + "=" * 60)
        print("SUMMARY")
        print("=" * 60)
        
        all_passed = True
        for test_name, passed in self.results.items():
            status = "PASS" if passed else "FAIL"
            if not passed:
                all_passed = False
            print(f"  [{status}] {test_name}")
        
        print(f"\nOverall: {'ALL TESTS PASSED' if all_passed else 'SOME TESTS FAILED'}")
        return all_passed

# Run the test framework
class TestConfig:
    vocab_size = 100
    hidden_size = 64

test_model = nn.Sequential(
    nn.Embedding(100, 64),
    nn.LayerNorm(64),
    nn.Linear(64, 100),
)

ref_model = nn.Sequential(
    nn.Embedding(100, 64),
    nn.LayerNorm(64),
    nn.Linear(64, 100),
)
ref_model.load_state_dict(test_model.state_dict())

framework = ModelTestFramework(test_model, TestConfig(), ref_model)
framework.run_all()

## 10. CI Integration Example

In [ ]:
# CI test script example

CI_TEST_SCRIPT = '''
# File: tests/test_my_model.py

import pytest
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


MODEL_NAME = "my-org/my-model-7b"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


@pytest.fixture(scope="module")
def hf_model():
    """Load HuggingFace model for reference."""
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16
    ).to(DEVICE).eval()
    return model


@pytest.fixture(scope="module")
def vllm_model():
    """Load vLLM model."""
    from vllm import LLM
    return LLM(model=MODEL_NAME, dtype="float16")


@pytest.fixture(scope="module")
def tokenizer():
    return AutoTokenizer.from_pretrained(MODEL_NAME)


class TestModelCorrectness:
    """Test suite for model implementation correctness."""
    
    @pytest.mark.parametrize("prompt", [
        "Hello, how are you?",
        "The capital of France is",
        "def fibonacci(n):",
    ])
    def test_greedy_decoding_matches(self, hf_model, vllm_model, tokenizer, prompt):
        """Greedy decoding should produce identical outputs."""
        # HuggingFace
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
        with torch.no_grad():
            hf_output = hf_model.generate(
                input_ids, max_new_tokens=20, do_sample=False
            )
        hf_text = tokenizer.decode(hf_output[0])
        
        # vLLM
        from vllm import SamplingParams
        params = SamplingParams(temperature=0, max_tokens=20)
        vllm_output = vllm_model.generate([prompt], params)
        vllm_text = prompt + vllm_output[0].outputs[0].text
        
        assert hf_text == vllm_text, f"Mismatch:\nHF: {hf_text}\nvLLM: {vllm_text}"
    
    def test_logits_close(self, hf_model, tokenizer):
        """Logits should be numerically close."""
        prompt = "The meaning of life is"
        input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)
        
        with torch.no_grad():
            hf_logits = hf_model(input_ids).logits
        
        # Compare with vLLM logits (need internal access)
        # This tests the model implementation directly
        assert not torch.isnan(hf_logits).any()
        assert not torch.isinf(hf_logits).any()
    
    def test_no_regression_on_benchmarks(self):
        """Model should not regress on key benchmarks."""
        # This would run evaluation benchmarks
        # e.g., MMLU, HumanEval, GSM8K
        pass


class TestEdgeCases:
    """Test edge cases."""
    
    def test_empty_prompt(self, vllm_model):
        """Model should handle empty prompt gracefully."""
        from vllm import SamplingParams
        params = SamplingParams(temperature=0, max_tokens=5)
        output = vllm_model.generate([""], params)
        assert len(output) == 1
    
    def test_max_length(self, vllm_model):
        """Model should stop at max_tokens."""
        from vllm import SamplingParams
        params = SamplingParams(temperature=0, max_tokens=1)
        output = vllm_model.generate(["Hello"], params)
        assert len(output[0].outputs[0].token_ids) == 1
    
    def test_batch_consistency(self, vllm_model):
        """Same prompt in batch should give same output."""
        from vllm import SamplingParams
        params = SamplingParams(temperature=0, max_tokens=10)
        outputs = vllm_model.generate(["Hello"] * 3, params)
        texts = [o.outputs[0].text for o in outputs]
        assert all(t == texts[0] for t in texts)
'''

print(CI_TEST_SCRIPT)

## 11. Exercises

### Exercise 1: Write a Complete Test Suite
Create a test suite for the model from Chapter 8.2 that includes:
- Unit tests for each layer
- Weight loading verification
- Output comparison with a reference
- Edge case handling

### Exercise 2: Implement a Tolerance Analyzer
Write a tool that takes two model implementations, runs them on the same inputs, and recommends appropriate tolerance thresholds.

### Exercise 3: Benchmark Test
Create a benchmark that measures both correctness and performance, tracking how implementation changes affect both.

In [ ]:
# Exercise 2: Tolerance Analyzer

def analyze_tolerances(
    model_a: nn.Module,
    model_b: nn.Module,
    vocab_size: int,
    n_samples: int = 50,
):
    """Analyze and recommend tolerances for two model implementations."""
    print("Tolerance Analysis")
    print("=" * 50)
    
    abs_diffs = []
    rel_diffs = []
    cos_sims = []
    
    for i in range(n_samples):
        seq_len = np.random.randint(1, 64)
        x = torch.randint(0, vocab_size, (1, seq_len))
        
        with torch.no_grad():
            out_a = model_a(x).float()
            out_b = model_b(x).float()
        
        abs_diff = (out_a - out_b).abs()
        abs_diffs.append(abs_diff.max().item())
        
        rel_diff = abs_diff / (out_a.abs() + 1e-8)
        rel_diffs.append(rel_diff.max().item())
        
        cos_sim = torch.nn.functional.cosine_similarity(
            out_a.flatten().unsqueeze(0),
            out_b.flatten().unsqueeze(0)
        ).item()
        cos_sims.append(cos_sim)
    
    abs_diffs = np.array(abs_diffs)
    rel_diffs = np.array(rel_diffs)
    cos_sims = np.array(cos_sims)
    
    print(f"\nAbsolute difference:")
    print(f"  Mean: {abs_diffs.mean():.6f}")
    print(f"  Max:  {abs_diffs.max():.6f}")
    print(f"  P99:  {np.percentile(abs_diffs, 99):.6f}")
    
    print(f"\nRelative difference:")
    print(f"  Mean: {rel_diffs.mean():.6f}")
    print(f"  Max:  {rel_diffs.max():.6f}")
    print(f"  P99:  {np.percentile(rel_diffs, 99):.6f}")
    
    print(f"\nCosine similarity:")
    print(f"  Mean: {cos_sims.mean():.8f}")
    print(f"  Min:  {cos_sims.min():.8f}")
    
    # Recommendations
    recommended_atol = np.percentile(abs_diffs, 99) * 2
    recommended_rtol = np.percentile(rel_diffs, 99) * 2
    
    print(f"\nRecommended tolerances (2x P99):")
    print(f"  atol: {recommended_atol:.6f}")
    print(f"  rtol: {recommended_rtol:.6f}")

# Demo
model_a = nn.Sequential(nn.Embedding(100, 64), nn.Linear(64, 100))
model_b = nn.Sequential(nn.Embedding(100, 64), nn.Linear(64, 100))
model_b.load_state_dict(model_a.state_dict())

analyze_tolerances(model_a, model_b, vocab_size=100)

## Summary

In this notebook, we covered a comprehensive testing strategy:

1. **Testing Pyramid**: From unit tests to quality benchmarks
2. **Tolerance Thresholds**: FP32 (1e-5) -> FP16 (1e-3) -> BF16 (1e-2) -> INT4 (0.5)
3. **Unit Tests**: Layer shapes, QKV split, RMSNorm, RoPE properties
4. **Weight Loading**: All weights loaded, no zeros, correct shapes
5. **Logit Comparison**: Absolute/relative diff, cosine similarity, top-k agreement
6. **Greedy Decoding**: Token-by-token comparison with reference
7. **Edge Cases**: Single token, max length, special tokens, determinism
8. **Perplexity**: Should match within 0.1%
9. **Automated Framework**: Reusable test infrastructure
10. **CI Integration**: pytest-based test scripts

### Next: Chapter 8.8 -- Performance Tuning Your Model Implementation